# Project 5: Roadside Assistant for Hidden Pedestrians and Cyclists (V2X / I2V)

A roadside camera (RSU) watches a crossing that an approaching vehicle cannot see because a parked vehicle blocks its view. The RSU detects pedestrians and cyclists, converts each detection into **world coordinates**, packs the result into a **compact message** stamped with the **shared simulation time**, and sends it to the vehicle over **MQTT**. The vehicle **fuses** the warning with its own camera, shows an **alert**, and **slows down**. Every scenario is run with and without assistance, and the notebook measures **warning lead time**, **speed reduction** and **near misses**.

| Cell | Content | Project requirement |
|---|---|---|
| 1 | Imports, configuration, connection | |
| 2 | Camera geometry: pixel to world, world to pixel | world coordinates |
| 3 | YOLOv8 detector for pedestrians and cyclists | VRU detector |
| 4 | Tracker (stable IDs and velocity) | compact estimate |
| 5 | V2X message format, MQTT link, receiver with shared timestamps | MQTT/V2X messaging, shared timestamps |
| 6 | Vehicle side: path check, fusion, slow-down policy, alert display | warning receiver, fusion, credible alert |
| 7 | Scenario runner with metrics | comparison with and without assistance |
| 8 | Scenario definitions and runs, results table | at least two occlusion scenarios |
| 9 | Speed plots for the report | results |

**Setup (once, in the Anaconda Prompt):**
```
conda activate carla916
pip install ultralytics paho-mqtt
```
**Optional, for a real MQTT broker:** install Eclipse Mosquitto and start it (`mosquitto -v`). If no broker is running on `localhost:1883`, the notebook automatically falls back to an in-process queue that carries the same serialized messages, and the results table records which link was used.

Run the cells in order. Start the CARLA server first.

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [ ]:
# ==============================================================================
# CELL 1: IMPORTS, CONFIGURATION AND CONNECTION
# ==============================================================================
import carla
import time
import math
import json
import random
import queue
import numpy as np
import cv2
import pandas as pd

# ---- Simulation ----------------------------------------------------------------
CARLA_HOST, CARLA_PORT = "localhost", 2000
MAP_NAME = "Town03"
FIXED_DT = 0.04               # simulation step [s] -> 25 Hz, synchronous mode
MAX_TICKS = 750               # at most 30 s of simulated time per run
DETECTION_INTERVAL = 0.2      # both detectors run every 0.2 s of simulated time (5 Hz)
ROLE_NAME = "roadside_assist" # tag on every actor we spawn, used to clean up leftovers

# ---- Cameras ---------------------------------------------------------------------
IMG_W, IMG_H = 720, 480
EGO_CAM_FOV = 100.0
RSU_CAM_FOV = 110.0
EGO_CAM_OFFSET = carla.Transform(carla.Location(x=2.0, z=1.3))
EGO_SPAWN = carla.Transform(carla.Location(x=103.0, y=133.0, z=1.0))
RSU_ID = "RSU-1"
RSU_TRANSFORM = carla.Transform(carla.Location(x=135.0, y=143.0, z=2.5),
                                carla.Rotation(pitch=-10.0, yaw=-110.0, roll=0.0))

# ---- V2X link ----------------------------------------------------------------------
MQTT_HOST, MQTT_PORT = "localhost", 1883
MQTT_TOPIC = "v2x/rsu1/vru"
V2X_DELAY_S = 0.10            # emulated RSU processing + radio delay, applied on the simulation clock
V2X_PACKET_LOSS = 0.0         # probability that a message is lost (0.0 = no loss)
OBJECT_TIMEOUT_S = 1.0        # forget an object that has not been reported for this long

# ---- Vehicle policy ------------------------------------------------------------------
PATH_HALF_WIDTH = 2.0         # m: a VRU closer than this to the ego's path line is "in path"
PREDICTION_HORIZON_S = 6.0    # s: only VRUs that enter the path within this time are considered
SAFETY_TIME_S = 1.0           # s: buffer when checking whether VRU and ego would meet in the path
MIN_ASSUMED_SPEED = 3.0       # m/s: a stopped/slow ego is assumed to drive at least this fast
WARNING_RANGE = 50.0          # m: objects farther ahead than this are ignored
CRUISE_SPEED_KMH = 30.0       # normal driving speed of the ego vehicle
V2X_TARGET_SPEED_KMH = 15.0   # RSU-only warning: the car keeps driving, but slows down to this speed
V2X_OVERRIDE_BRAKE = 0.3      # RSU-only warning brakes directly only if the needed brake exceeds this
STOP_MARGIN = 2.0             # m: braking aims to stop this far before the VRU
MAX_DECEL = 8.0               # m/s^2: approximate deceleration at brake = 1.0
RESUME_HOLD_S = 1.5           # s of simulated time without a threat before normal driving resumes

# ---- Metrics ---------------------------------------------------------------------------
NEAR_MISS_GAP = 2.0           # m between the ego's body and a VRU -> near miss
COLLISION_GAP = 0.3           # m -> counted as a collision (in addition to the collision sensor)

STATUS_EVERY_S = 1.0          # print a status line this often (simulated seconds); 0 = off

FONT = cv2.FONT_HERSHEY_SIMPLEX


def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Place the editor spectator camera behind an actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))


def safe_destroy(actors):
    """Destroy actors, ignoring ones that are already gone."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass


client = carla.Client(CARLA_HOST, CARLA_PORT)
client.set_timeout(30.0)
print(f"Connected to CARLA. Client {client.get_client_version()} | Server {client.get_server_version()}")

## Cell 2: Camera geometry (pixel to world coordinates)

A detector gives a box in **pixels**. The vehicle needs to know **where on the road** the person is, in the same **world coordinates** CARLA uses for everything else. Because the RSU camera is fixed and its pose is known, one pixel can be turned into one point on the road:

1. The **intrinsics** come from the image size and field of view: the focal length is `f = W / (2 * tan(FOV/2))`, and the image centre is `(W/2, H/2)`.
2. The **bottom centre** of a detection box is where the person's feet touch the ground.
3. That pixel defines a **ray** leaving the camera. The camera's pose (the **extrinsics**, a 4x4 matrix built the same way as `carla.Transform.get_matrix()`) rotates the ray into world coordinates.
4. The ray is intersected with the **ground plane** `z = ground_z`, which gives the world `(x, y)` of the person.

`world_to_pixel` does the opposite. The vehicle uses it to draw the RSU-reported position into its own camera image, even when the person is hidden behind the parked van.

In [ ]:
# ==============================================================================
# CELL 2: CAMERA GEOMETRY - PIXEL <-> WORLD COORDINATES
# ==============================================================================

def pose_matrix(x, y, z, pitch, yaw, roll):
    """Local-to-world 4x4 matrix, same convention as carla.Transform.get_matrix() (angles in degrees)."""
    cy, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cr, sr = math.cos(math.radians(roll)), math.sin(math.radians(roll))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    return np.array([
        [cp * cy, cy * sp * sr - sy * cr, -cy * sp * cr - sy * sr, x],
        [cp * sy, sy * sp * sr + cy * cr, -sy * sp * cr + cy * sr, y],
        [sp,      -cp * sr,               cp * cr,                 z],
        [0.0,     0.0,                    0.0,                     1.0]])


def transform_matrix(tf):
    """4x4 matrix of a carla.Transform."""
    return pose_matrix(tf.location.x, tf.location.y, tf.location.z,
                       tf.rotation.pitch, tf.rotation.yaw, tf.rotation.roll)


def focal_length(width, fov_deg):
    """Pinhole focal length in pixels from image width and horizontal field of view."""
    return width / (2.0 * math.tan(math.radians(fov_deg) / 2.0))


def pixel_to_ground(u, v, cam_m, fov_deg, ground_z, width=IMG_W, height=IMG_H):
    """
    Back-project pixel (u, v) onto the ground plane z = ground_z.
    Returns world (x, y), or None if the pixel is above the horizon.
    """
    f = focal_length(width, fov_deg)
    # Ray in the camera's local frame (CARLA/UE4 axes: x forward, y right, z up)
    d_local = np.array([1.0, (u - width / 2.0) / f, -(v - height / 2.0) / f])
    d = cam_m[:3, :3] @ d_local          # rotate the ray into world coordinates
    origin = cam_m[:3, 3]
    if d[2] >= -1e-6:                    # ray does not point down -> never hits the ground
        return None
    t = (ground_z - origin[2]) / d[2]
    if t <= 0:
        return None
    p = origin + t * d
    return float(p[0]), float(p[1])


def world_to_pixel(xyz, cam_m, fov_deg, width=IMG_W, height=IMG_H):
    """Project a world point into the image. Returns (u, v) or None if behind the camera."""
    p = np.linalg.inv(cam_m) @ np.array([xyz[0], xyz[1], xyz[2], 1.0])
    if p[0] <= 0.1:
        return None
    f = focal_length(width, fov_deg)
    u = f * p[1] / p[0] + width / 2.0
    v = f * (-p[2]) / p[0] + height / 2.0
    return int(round(u)), int(round(v))


def to_ego_frame(x, y, ego_x, ego_y, ego_yaw_deg):
    """World point -> (longitudinal, lateral) offset in the ego vehicle's frame. lateral > 0 = right."""
    c, s = math.cos(math.radians(ego_yaw_deg)), math.sin(math.radians(ego_yaw_deg))
    dx, dy = x - ego_x, y - ego_y
    return dx * c + dy * s, -dx * s + dy * c


def gap_to_ego_body(lon, lat, ext_x, ext_y):
    """Distance from a point (in the ego frame) to the ego's bounding box. 0 means touching."""
    return math.hypot(max(abs(lon) - ext_x, 0.0), max(abs(lat) - ext_y, 0.0))


print("Geometry helpers defined.")

## Cell 3: Detector for pedestrians and cyclists (YOLOv8n)

YOLOv8n (pre-trained on COCO) runs on both cameras. Only two COCO classes are kept: **person (0)** and **bicycle (1)**. Every other class is dropped, so cars and vans can no longer be mistaken for pedestrians.

A cyclist appears to YOLO as a *person* on top of a *bicycle*. `merge_person_bicycle` pairs each bicycle box with the person box sitting on it and reports one **cyclist**. Persons without a bicycle become **pedestrians**. The "foot point" (bottom centre of the box) is what Cell 2 projects onto the road.

Images are passed to YOLOv8 in OpenCV's BGR order, which is what Ultralytics expects for NumPy arrays.

In [ ]:
# ==============================================================================
# CELL 3: VRU DETECTOR - PEDESTRIANS AND CYCLISTS (YOLOv8n)
# ==============================================================================
import torch
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")
model.to(DEVICE)

COCO_PERSON, COCO_BICYCLE = 0, 1
PERSON_CONF = 0.6     # minimum confidence for a person
BICYCLE_CONF = 0.4    # bicycles are thinner and get lower scores, so the bar is lower

# Warm-up: the first inference is slow; do it now instead of during the first scenario
model(np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8), verbose=False)
print(f"YOLOv8n loaded on {DEVICE}.")


def merge_person_bicycle(persons, bikes):
    """
    persons, bikes: lists of (box, conf) with box = (x1, y1, x2, y2) in pixels.
    Returns a list of dicts: cls ('pedestrian' | 'cyclist'), conf, box, foot (u, v).
    """
    out, used = [], set()
    for (bx1, by1, bx2, by2), bconf in bikes:
        rider = None
        for i, ((px1, py1, px2, py2), _) in enumerate(persons):
            if i in used:
                continue
            pcx = (px1 + px2) / 2.0
            # the rider's centre lies above the bicycle and their lower edge is inside it
            if bx1 - 5 <= pcx <= bx2 + 5 and by1 - (by2 - by1) <= py2 <= by2 + 5:
                rider = i
                break
        box, conf = (bx1, by1, bx2, by2), bconf
        if rider is not None:
            used.add(rider)
            (px1, py1, px2, py2), pconf = persons[rider]
            box = (min(bx1, px1), min(by1, py1), max(bx2, px2), max(by2, py2))
            conf = max(bconf, pconf)
        out.append(dict(cls="cyclist", conf=conf, box=box, foot=((bx1 + bx2) / 2.0, by2)))
    for i, ((px1, py1, px2, py2), pconf) in enumerate(persons):
        if i not in used:
            out.append(dict(cls="pedestrian", conf=pconf, box=(px1, py1, px2, py2),
                            foot=((px1 + px2) / 2.0, py2)))
    return out


def detect_vrus(img_bgr):
    """Run YOLOv8 on one BGR frame and return pedestrian/cyclist detections (see merge_person_bicycle)."""
    res = model(img_bgr, classes=[COCO_PERSON, COCO_BICYCLE],
                conf=min(PERSON_CONF, BICYCLE_CONF), verbose=False)[0]
    persons, bikes = [], []
    for b in res.boxes:
        cls, conf = int(b.cls[0]), float(b.conf[0])
        box = tuple(float(v) for v in b.xyxy[0])
        if cls == COCO_PERSON and conf >= PERSON_CONF:
            persons.append((box, conf))
        elif cls == COCO_BICYCLE and conf >= BICYCLE_CONF:
            bikes.append((box, conf))
    return merge_person_bicycle(persons, bikes)


def draw_box(img, box, label, color):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.putText(img, label, (x1, max(15, y1 - 8)), FONT, 0.5, color, 2)

## Cell 4: Tracker

A single detection says *where* someone is, but not *where they are going*. The tracker links detections from consecutive detection cycles by picking the nearest previous position within 2.5 m. This gives each VRU a **stable ID** and a **velocity** `(vx, vy)`, computed from how far it moved since the last cycle and lightly smoothed. The vehicle uses the velocity to predict whether a VRU will walk into its path. Tracks that are not seen for `OBJECT_TIMEOUT_S` are dropped.

The same tracker runs on both sides: at the RSU on its detections, and in the vehicle on its own camera's detections.

In [ ]:
# ==============================================================================
# CELL 4: SIMPLE MULTI-OBJECT TRACKER (WORLD COORDINATES)
# ==============================================================================

class SimpleTracker:
    """Nearest-neighbour tracker in world coordinates: stable IDs and velocity estimates."""

    def __init__(self, gate_m=2.5, timeout_s=OBJECT_TIMEOUT_S, alpha=0.5):
        self.gate, self.timeout, self.alpha = gate_m, timeout_s, alpha
        self.tracks = {}
        self.next_id = 1

    def update(self, dets, t):
        """
        dets: list of dicts with x, y, cls, conf (world coordinates) observed at time t.
        Returns copies of the tracks that were updated at time t.
        """
        # forget tracks that have not been seen recently
        self.tracks = {k: tr for k, tr in self.tracks.items() if t - tr["t"] <= self.timeout}

        # greedy association: closest (detection, track) pairs first
        pairs = sorted(
            ((math.hypot(d["x"] - tr["x"], d["y"] - tr["y"]), i, k)
             for i, d in enumerate(dets) for k, tr in self.tracks.items()),
            key=lambda p: p[0])
        used_d, used_t, updated = set(), set(), []
        for dist, i, k in pairs:
            if dist > self.gate:
                break
            if i in used_d or k in used_t:
                continue
            used_d.add(i)
            used_t.add(k)
            tr, d = self.tracks[k], dets[i]
            dt = t - tr["t"]
            if dt > 1e-3:
                vx, vy = (d["x"] - tr["x"]) / dt, (d["y"] - tr["y"]) / dt
                a = self.alpha if tr["n"] > 1 else 1.0   # second sighting: take the raw velocity
                tr["vx"] = a * vx + (1 - a) * tr["vx"]
                tr["vy"] = a * vy + (1 - a) * tr["vy"]
            tr.update(x=d["x"], y=d["y"], t=t, cls=d["cls"], conf=d["conf"], n=tr["n"] + 1)
            updated.append(tr)

        # unmatched detections start new tracks
        for i, d in enumerate(dets):
            if i in used_d:
                continue
            tr = dict(id=self.next_id, x=d["x"], y=d["y"], vx=0.0, vy=0.0,
                      t=t, cls=d["cls"], conf=d["conf"], n=1)
            self.tracks[self.next_id] = tr
            self.next_id += 1
            updated.append(tr)
        return [dict(tr) for tr in updated]


print("Tracker defined.")

## Cell 5: V2X message, MQTT link and shared timestamps

**Message design.** Each RSU detection cycle produces one compact JSON message, with short keys and rounded values. A typical message with one pedestrian is about 110 bytes:

```json
{"v":1,"rsu":"RSU-1","seq":42,"t":12.4,
 "objs":[{"id":3,"c":"ped","x":130.82,"y":141.2,"vx":0.02,"vy":-3.41,"cf":0.91}]}
```

| Field | Meaning |
|---|---|
| `v` | message format version |
| `rsu` | sender ID |
| `seq` | sequence number (detects lost messages) |
| `t` | **capture time on the shared CARLA simulation clock** [s] |
| `objs[].id` | track ID, stable across messages |
| `objs[].c` | class: `ped` or `cyc` |
| `objs[].x, y` | position in CARLA world coordinates [m], 1 cm resolution |
| `objs[].vx, vy` | velocity [m/s] |
| `objs[].cf` | detector confidence |

**Transport.** The RSU publishes to the MQTT topic `v2x/rsu1/vru`, and the vehicle subscribes to it. Both use separate MQTT clients, as two separate devices would. If no broker is reachable, the same bytes go through an in-process queue instead.

**Shared timestamps.** Both sides use CARLA's **simulation clock**: the RSU stamps the time its camera captured the frame, and the vehicle reads the same clock. This gives three things:
1. **Emulated delay.** The receiver holds a message until `sim_now >= t + V2X_DELAY_S`, so the vehicle never uses information earlier than it could arrive in reality.
2. **Measured latency** = `sim time at delivery - t`, per message.
3. **Latency compensation.** The vehicle moves each reported object forward by `velocity x message age` before using it.

The real MQTT transport time on the computer (wall clock) is measured separately.

In [ ]:
# ==============================================================================
# CELL 5: V2X MESSAGE FORMAT, MQTT LINK AND RECEIVER (SHARED SIMULATION CLOCK)
# ==============================================================================
MSG_VERSION = 1
CLS_CODE = {"pedestrian": "ped", "cyclist": "cyc"}
CODE_CLS = {v: k for k, v in CLS_CODE.items()}


def encode_vru_message(rsu_id, seq, t_capture, tracks):
    """RSU side: tracks -> compact JSON bytes."""
    msg = {"v": MSG_VERSION, "rsu": rsu_id, "seq": seq, "t": round(t_capture, 3),
           "objs": [{"id": tr["id"], "c": CLS_CODE[tr["cls"]],
                     "x": round(tr["x"], 2), "y": round(tr["y"], 2),
                     "vx": round(tr["vx"], 2), "vy": round(tr["vy"], 2),
                     "cf": round(tr["conf"], 2)} for tr in tracks]}
    return json.dumps(msg, separators=(",", ":")).encode("utf-8")


def decode_vru_message(payload):
    """Vehicle side: JSON bytes -> dict with readable object fields."""
    msg = json.loads(payload.decode("utf-8"))
    if msg.get("v") != MSG_VERSION:
        raise ValueError(f"unsupported message version {msg.get('v')}")
    msg["objs"] = [dict(id=o["id"], cls=CODE_CLS.get(o["c"], "pedestrian"),
                        x=o["x"], y=o["y"], vx=o["vx"], vy=o["vy"], conf=o["cf"])
                   for o in msg["objs"]]
    return msg


class V2XLink:
    """
    RSU -> vehicle channel. Uses MQTT (paho-mqtt) when a broker is reachable,
    otherwise an in-process queue. Serialized bytes travel either way.
    """

    def __init__(self, host=MQTT_HOST, port=MQTT_PORT, topic=MQTT_TOPIC):
        self.topic = topic
        self.inbox = queue.Queue()
        self.sent_wall = {}          # seq -> wall-clock send time, to measure transport time
        self.mode = "local-queue"
        self._pub = self._sub = None
        try:
            import paho.mqtt.client as mqtt
            suffix = random.randint(0, 99999)
            self._sub = self._make_client(mqtt, f"ego-vehicle-{suffix}")
            self._sub.on_message = self._on_message
            self._sub.connect(host, port, keepalive=30)
            self._sub.subscribe(topic, qos=0)
            self._sub.loop_start()
            self._pub = self._make_client(mqtt, f"rsu-{suffix}")
            self._pub.connect(host, port, keepalive=30)
            self._pub.loop_start()
            time.sleep(0.5)          # let the subscription become active
            self.mode = "mqtt"
        except Exception as e:
            self.close()
            self._pub = self._sub = None
            print(f"[V2X] MQTT broker not reachable ({e}). Using the in-process queue instead.")
        print(f"[V2X] link mode: {self.mode}")

    @staticmethod
    def _make_client(mqtt, client_id):
        try:
            return mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=client_id)  # paho-mqtt 2.x
        except AttributeError:
            return mqtt.Client(client_id=client_id)                                     # paho-mqtt 1.x

    def _on_message(self, client, userdata, msg):
        self.inbox.put((msg.payload, time.perf_counter()))

    def publish(self, payload, seq):
        self.sent_wall[seq] = time.perf_counter()
        if self.mode == "mqtt":
            self._pub.publish(self.topic, payload, qos=0)
        else:
            self.inbox.put((payload, time.perf_counter()))

    def receive_all(self):
        out = []
        while True:
            try:
                out.append(self.inbox.get_nowait())
            except queue.Empty:
                return out

    def close(self):
        for c in (self._pub, self._sub):
            if c is not None:
                try:
                    c.loop_stop()
                    c.disconnect()
                except Exception:
                    pass


class V2XReceiver:
    """
    Vehicle-side warning receiver. Holds each message until the shared simulation
    clock reaches capture time + V2X_DELAY_S, then makes its objects available.
    """

    def __init__(self, link):
        self.link = link
        self.pending = []            # (deliver_at_sim_time, message)
        self.objects = []            # objects from the latest delivered message
        self.last_t = None
        self.stats = dict(received=0, lost=0, delivered=0, bytes=[], sim_latency=[], wall_ms=[])

    def poll(self, sim_now):
        for payload, recv_wall in self.link.receive_all():
            self.stats["received"] += 1
            if random.random() < V2X_PACKET_LOSS:
                self.stats["lost"] += 1
                continue
            msg = decode_vru_message(payload)
            self.stats["bytes"].append(len(payload))
            sent = self.link.sent_wall.pop(msg["seq"], None)
            if sent is not None:
                self.stats["wall_ms"].append((recv_wall - sent) * 1000.0)
            self.pending.append((msg["t"] + V2X_DELAY_S, msg))

        ready = sorted((p for p in self.pending if p[0] <= sim_now + 1e-6), key=lambda p: p[0])
        self.pending = [p for p in self.pending if p[0] > sim_now + 1e-6]
        for _, msg in ready:
            self.stats["delivered"] += 1
            self.stats["sim_latency"].append(sim_now - msg["t"])
            self.objects = [dict(o, t=msg["t"], source=msg["rsu"]) for o in msg["objs"]]
            self.last_t = msg["t"]

        if self.last_t is not None and sim_now - self.last_t > OBJECT_TIMEOUT_S:
            self.objects = []        # no fresh information: do not act on stale objects
        return list(self.objects)


print("V2X message format, link and receiver defined.")

## Cell 6: Vehicle side: fusion, slow-down policy and alert

**Is the VRU in my path when I get there?** Every object, whether from the RSU or the vehicle's own camera, is converted into the ego frame: `lon` is the distance ahead, and `lat` is the sideways offset from the ego's path line. Two time windows are then compared:
- **VRU window:** from its sideways speed, when it enters and leaves the path corridor (`|lat| < PATH_HALF_WIDTH`);
- **ego window:** from the car's speed, when the car reaches and passes that spot. A stopped car is assumed to drive at least `MIN_ASSUMED_SPEED`.

An object is **relevant** only if the two windows overlap, within `SAFETY_TIME_S`. So a pedestrian standing on the pavement is ignored, one walking away is ignored, and one who will have finished crossing before the car arrives is ignored. That last case matters: without it, a stopped car would wait forever for every pedestrian anywhere ahead.

**Fusion and policy.**

| Alert level | When | Action |
|---|---|---|
| **DANGER** | the vehicle's own camera sees a relevant VRU | brake just hard enough to stop `STOP_MARGIN` before it: required deceleration `v^2 / (2 * gap)`, brake between 0.3 and 1.0. Labelled `camera+RSU` if the RSU reports the same object within 2.5 m (confirmed), else `camera` |
| **CAUTION** | only the RSU reports a relevant VRU (hidden from the camera) | the car **keeps driving, but slows down to `V2X_TARGET_SPEED_KMH`**. Only if the hidden VRU is so close that the required brake exceeds `V2X_OVERRIDE_BRAKE` does the car brake directly |
| **NONE** | nothing relevant | normal driving at `CRUISE_SPEED_KMH` (after `RESUME_HOLD_S` without a threat) |

In the **baseline** runs, RSU messages are still received and logged, which is needed to measure the lead time, but the policy ignores them.

**Credible alert.** The ego camera window shows a coloured banner with level, class, distance, source and message age (for example *"CAUTION: HIDDEN PEDESTRIAN 17.3 m AHEAD (RSU, 120 ms old)"*). It also draws a circle where the RSU says the hidden VRU is, even when it's behind the van. The same position is marked in the CARLA world.


**Driver.** `LaneFollower` replaces CARLA's Traffic Manager autopilot. It steers towards a point on the lane centre a few metres ahead (pure pursuit, using the map's waypoints; at junctions it goes straight) and controls throttle and brake to reach the target speed: `CRUISE_SPEED_KMH` normally, `V2X_TARGET_SPEED_KMH` after an RSU warning. It ignores traffic lights and other road users, so every reaction to a VRU comes from the policy above.

In [ ]:
# ==============================================================================
# CELL 6: VEHICLE SIDE - PATH CHECK, FUSION, SLOW-DOWN POLICY, ALERT DISPLAY
# ==============================================================================

def assess_threat(obj, ego_tf, ego_speed, sim_now, ego_ext_x=2.4):
    """
    obj: dict with x, y, vx, vy, cls, conf and t (time of the observation).
    Moves the object forward to 'now' (latency compensation), expresses it in the
    ego frame, and decides whether it will be in the ego's path at the time the ego
    gets there (time-window check).
    """
    age = max(0.0, sim_now - obj.get("t", sim_now))
    x = obj["x"] + obj["vx"] * age
    y = obj["y"] + obj["vy"] * age
    yaw = ego_tf.rotation.yaw
    lon, lat = to_ego_frame(x, y, ego_tf.location.x, ego_tf.location.y, yaw)
    v_lat = -obj["vx"] * math.sin(math.radians(yaw)) + obj["vy"] * math.cos(math.radians(yaw))
    inf = float("inf")

    # 1) When is the VRU inside the ego's path corridor (|lateral| < PATH_HALF_WIDTH)?
    if abs(v_lat) < 0.2:                                     # (almost) not moving sideways
        vru_in, vru_out = (0.0, inf) if abs(lat) < PATH_HALF_WIDTH else (inf, inf)
    else:
        ta = (-PATH_HALF_WIDTH - lat) / v_lat
        tb = (PATH_HALF_WIDTH - lat) / v_lat
        vru_in, vru_out = max(min(ta, tb), 0.0), max(ta, tb)
        if vru_out < 0:                                      # already crossed and walking away
            vru_in, vru_out = inf, inf

    # 2) When does the ego occupy that spot? A stopped car is assumed to start moving,
    #    so its speed is taken as at least MIN_ASSUMED_SPEED.
    v_eff = max(ego_speed, MIN_ASSUMED_SPEED)
    ego_in = max(lon - ego_ext_x - 1.0, 0.0) / v_eff
    ego_out = (lon + ego_ext_x + 1.0) / v_eff

    # 3) Relevant if both time windows overlap (with a safety buffer)
    relevant = (0.0 < lon < WARNING_RANGE and vru_in <= PREDICTION_HORIZON_S
                and vru_in <= ego_out + SAFETY_TIME_S and vru_out >= ego_in - SAFETY_TIME_S)
    t_pred = min(max(lon, 0.0) / v_eff, PREDICTION_HORIZON_S)
    return dict(obj, x_now=x, y_now=y, lon=lon, lat=lat, lat_pred=lat + v_lat * t_pred,
                age=age, relevant=relevant)


def required_brake(speed_ms, lon, ego_ext_x):
    """Brake command needed to stop STOP_MARGIN before an object lon metres ahead of the ego centre."""
    gap = lon - ego_ext_x - STOP_MARGIN
    a_req = speed_ms ** 2 / (2.0 * max(gap, 0.5))
    return float(np.clip(a_req / MAX_DECEL, 0.0, 1.0))


def decide(local_threats, v2x_threats, v2x_enabled, speed_ms, ego_ext_x):
    """
    Fuse local and RSU information.
    Returns level, brake (> 0 means: brake now), and speed_cap_kmh
    (not None means: keep driving, but slow down to this speed).
    """
    local = [t for t in local_threats if t["relevant"]]
    remote = [t for t in v2x_threats if t["relevant"]] if v2x_enabled else []

    if local:
        th = min(local, key=lambda t: t["lon"])
        confirmed = any(math.hypot(th["x_now"] - r["x_now"], th["y_now"] - r["y_now"]) < 2.5 for r in remote)
        brake = max(0.3, required_brake(speed_ms, th["lon"], ego_ext_x))
        return dict(level="DANGER", brake=brake, speed_cap_kmh=None,
                    source="camera+RSU" if confirmed else "camera", threat=th)

    if remote:
        th = min(remote, key=lambda t: t["lon"])
        req = required_brake(speed_ms, th["lon"], ego_ext_x)
        # Normally the car just slows down to the target speed. Only if the hidden VRU is
        # so close that a real braking manoeuvre is needed does the RSU warning brake directly.
        brake = req if req >= V2X_OVERRIDE_BRAKE else 0.0
        return dict(level="CAUTION", brake=brake, speed_cap_kmh=V2X_TARGET_SPEED_KMH, source="RSU", threat=th)

    return dict(level="NONE", brake=0.0, speed_cap_kmh=None, source="", threat=None)


class LaneFollower:
    """
    Simple driver that replaces CARLA's Traffic Manager autopilot:
    - steering: pure pursuit towards a point on the lane centre, a few metres ahead (from map waypoints);
      at junctions it takes the straightest continuation;
    - speed: proportional throttle/brake towards a target speed.
    It ignores traffic lights, other vehicles and VRUs; every reaction to VRUs comes from our policy.
    """
    WHEELBASE = 2.9            # m (Tesla Model 3)

    def __init__(self, carla_map, vehicle=None):
        self.map = carla_map
        self.max_steer = math.radians(70.0)
        try:
            self.max_steer = math.radians(vehicle.get_physics_control().wheels[0].max_steer_angle)
        except Exception:
            pass

    def target_point(self, ego_tf, speed):
        wp = self.map.get_waypoint(ego_tf.location, project_to_road=True, lane_type=carla.LaneType.Driving)
        if wp is None:
            return None
        look = max(5.0, 0.8 * speed + 3.0)
        fwd, wfwd = ego_tf.get_forward_vector(), wp.transform.get_forward_vector()
        same_dir = fwd.x * wfwd.x + fwd.y * wfwd.y >= 0.0
        cands = wp.next(look) if same_dir else wp.previous(look)
        if not cands:
            return None

        def turn(w):   # yaw change relative to the current lane direction
            return abs((w.transform.rotation.yaw - wp.transform.rotation.yaw + 180.0) % 360.0 - 180.0)
        return min(cands, key=turn).transform.location

    def steer(self, ego_tf, speed):
        target = self.target_point(ego_tf, speed)
        if target is None:
            return 0.0
        lon, lat = to_ego_frame(target.x, target.y, ego_tf.location.x, ego_tf.location.y, ego_tf.rotation.yaw)
        d2 = lon * lon + lat * lat
        if d2 < 1e-3:
            return 0.0
        delta = math.atan2(2.0 * self.WHEELBASE * lat, d2)     # pure pursuit; lat > 0 = right = positive steer
        return float(np.clip(delta / self.max_steer, -1.0, 1.0))

    @staticmethod
    def speed_control(speed_ms, target_kmh):
        """Returns (throttle, brake) to approach the target speed."""
        err = target_kmh / 3.6 - speed_ms                       # m/s, > 0 means too slow
        if err < -0.8:
            return 0.0, float(np.clip(0.12 * -err, 0.05, 0.4))
        return float(np.clip(0.3 + 0.3 * err, 0.0, 0.85)), 0.0


LEVEL_COLOR_BGR = {"NONE": (0, 200, 0), "CAUTION": (0, 165, 255), "DANGER": (0, 0, 255)}
LEVEL_COLOR_CARLA = {"NONE": carla.Color(0, 200, 0), "CAUTION": carla.Color(255, 165, 0),
                     "DANGER": carla.Color(255, 0, 0)}


def draw_ego_hud(img, decision, v2x_enabled, speed_kmh, v2x_threats, cam_m, ground_z, ego_ext_x, link_mode):
    """Alert banner plus markers for RSU-reported objects (visible even when occluded)."""
    h, w = img.shape[:2]
    level = decision["level"]
    color = LEVEL_COLOR_BGR[level]
    cv2.rectangle(img, (0, 0), (w, 64), (20, 20, 20), -1)
    cv2.rectangle(img, (2, 2), (w - 3, 62), color, 2)
    if level == "NONE":
        line1 = f"{'V2X ASSISTED' if v2x_enabled else 'UNASSISTED'} | no VRU in path"
    else:
        th = decision["threat"]
        dist = max(th["lon"] - ego_ext_x, 0.0)
        what = th["cls"].upper()
        if level == "DANGER":
            line1 = f"DANGER: {what} {dist:.1f} m AHEAD - BRAKING ({decision['source']})"
        else:
            action = "BRAKING" if decision["brake"] > 0 else f"SLOWING TO {decision['speed_cap_kmh']:.0f} km/h"
            line1 = f"CAUTION: HIDDEN {what} {dist:.1f} m AHEAD (RSU, {th['age'] * 1000:.0f} ms old) - {action}"
    line2 = f"speed {speed_kmh:5.1f} km/h | brake {decision['brake']:.2f} | V2X link: {link_mode}"
    cv2.putText(img, line1, (12, 27), FONT, 0.55, color, 2)
    cv2.putText(img, line2, (12, 52), FONT, 0.5, (230, 230, 230), 1)

    if v2x_enabled:
        for th in v2x_threats:
            px = world_to_pixel((th["x_now"], th["y_now"], ground_z + 0.9), cam_m, EGO_CAM_FOV)
            if px is None or not (0 <= px[0] < w and 70 <= px[1] < h):
                continue
            c = LEVEL_COLOR_BGR["CAUTION"] if th["relevant"] else (180, 180, 180)
            cv2.circle(img, px, 16, c, 2)
            cv2.putText(img, f"RSU: {th['cls']}", (px[0] - 40, px[1] - 22), FONT, 0.5, c, 2)
    return img


print("Vehicle-side fusion, policy and HUD defined.")

## Cell 7: Scenario runner and metrics

One call runs one scenario in one mode (baseline or V2X). Per simulation tick, it:

1. advances the world (synchronous mode, 25 Hz) and reads the **shared simulation time**;
2. every 0.2 s: **RSU** detects, projects to the road, tracks, encodes and **publishes**; the **vehicle** detects with its own camera, projects and tracks;
3. every tick: the vehicle **receives** due RSU messages, checks path relevance, **decides**, computes steering, throttle and brake with our own lane-following driver, and updates the alert display;
4. logs speed, alert level and the ground-truth gap between the car body and the nearest VRU.

**Metrics** (all on the simulation clock):

| Metric | Definition |
|---|---|
| Warning lead time | time from the first relevant RSU warning available at the vehicle to the first relevant detection by the vehicle's own camera. Taken from the baseline run, where the vehicle's behaviour is not influenced by the warning |
| Lead distance | how much farther from the conflict point the vehicle was when the RSU warning arrived vs. when its own camera saw the VRU |
| Speed reduction | speed at the first reaction minus the lowest speed before the conflict point |
| Speed at closest approach | vehicle speed at the moment it was closest to a VRU |
| Near miss | no collision, but the gap between car body and VRU fell below `NEAR_MISS_GAP` |
| Collision | collision sensor hit a VRU, or gap below `COLLISION_GAP` |
| RSU localisation error | distance between the RSU's estimated position and the true VRU position (validation of the pixel-to-world conversion) |
| V2X latency / message size | measured per message |

**Driving.** The car is driven by our own `LaneFollower` (Cell 6) instead of CARLA's Traffic Manager autopilot, which gave no throttle in synchronous mode. Bicycles are moved kinematically along a straight line, so they cannot tip over or be pushed off course. Both make every run repeatable.

**Stability fixes** from earlier: the map loads only once, sensors are stopped before actors are destroyed, the server is always returned to asynchronous mode, and OpenCV windows are closed properly.

**Status output.** Once per simulated second a line shows speed, driver state (`CRUISE`, `SLOW`, `BRAKE`, `HOLD`), throttle/steer/brake, alert and the nearest VRU. If the car stands still for 3 s, a `[STUCK]` line explains why.

In [ ]:
# ==============================================================================
# CELL 7: SCENARIO RUNNER AND METRICS
# ==============================================================================

def tagged(bp):
    """Mark a blueprint with our role name so leftovers can be found and removed later."""
    if bp.has_attribute("role_name"):
        bp.set_attribute("role_name", ROLE_NAME)
    return bp


def first_blueprint(lib, names):
    """First blueprint that exists in this CARLA version, from a list of candidates."""
    for n in names:
        found = lib.filter(n)
        if len(found) > 0:
            return found[0]
    raise RuntimeError(f"None of these blueprints exist: {names}")


def spawn_with_retries(world, bp, x, y, z, yaw, tries=10):
    for k in range(tries):
        tf = carla.Transform(carla.Location(x=x + 0.2 * k, y=y + 0.1 * k, z=z), carla.Rotation(yaw=yaw))
        actor = world.try_spawn_actor(bp, tf)
        if actor is not None:
            return actor
    return None


def cleanup_leftovers(world):
    """Remove actors left behind by an earlier run that crashed before its cleanup."""
    leftovers = [a for a in world.get_actors() if a.attributes.get("role_name") == ROLE_NAME]
    for a in leftovers:
        if a.type_id.startswith("sensor."):
            try:
                a.stop()
            except Exception:
                pass
    safe_destroy(leftovers)
    if leftovers:
        print(f"Removed {len(leftovers)} leftover actors from an earlier run.")


def estimate_ground_z(world, x, y, fallback):
    """Road height at (x, y), used as the ground plane for pixel-to-world projection."""
    try:
        lp = world.ground_projection(carla.Location(x=x, y=y, z=fallback + 10.0), 30.0)
        if lp is not None:
            return lp.location.z
    except Exception:
        pass
    return fallback


def make_camera_callback(store, key):
    """Keep the latest frame with its simulation timestamp and the camera pose at capture."""
    def callback(image):
        arr = np.frombuffer(image.raw_data, dtype=np.uint8).reshape((image.height, image.width, 4))
        store[key] = (arr[:, :, :3].copy(), image.timestamp, transform_matrix(image.transform))
    return callback


def run_scenario(world, sc, v2x_enabled, link):
    mode = "V2X" if v2x_enabled else "baseline"
    print(f"\n{'=' * 80}\n{sc['id']}: {sc['name']} | {mode}\n{'=' * 80}")

    world.set_weather(getattr(carla.WeatherParameters, sc["weather"]))
    lib = world.get_blueprint_library()
    spectator = world.get_spectator()

    original_settings = world.get_settings()
    original_settings.synchronous_mode = False   # never leave the server frozen after a run
    original_settings.fixed_delta_seconds = None
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = FIXED_DT
    world.apply_settings(settings)

    actors, vrus, vru_ids, collisions = [], [], set(), []
    ego = ego_cam = rsu_cam = col_sensor = None
    frames = {"ego": None, "rsu": None}
    link.receive_all()                     # discard messages left from a previous run
    link.sent_wall.clear()
    receiver = V2XReceiver(link)
    rsu_tracker, ego_tracker = SimpleTracker(), SimpleTracker()
    log, loc_errors = [], []
    events = dict(v2x_first=None, local_first=None, reaction=None)
    example_msg = None
    seq = 0
    last_threat_t, hold_brake = -1e9, 0.0
    last_caution_t, state = -1e9, "CRUISE"
    bikes = []                             # (actor, start transform) of kinematically moved bicycles
    local_tracks = []
    ext_x = ext_y = 1.0

    try:
        # ---------------- spawn actors ----------------
        ego = world.try_spawn_actor(tagged(lib.filter("vehicle.tesla.model3")[0]), EGO_SPAWN)
        if ego is None:
            raise RuntimeError("Ego vehicle could not spawn (spawn point blocked?).")
        actors.append(ego)

        ox, oy = sc["occluder_pos"]
        occluder = world.try_spawn_actor(tagged(first_blueprint(lib, sc["occluder"])),
                                         carla.Transform(carla.Location(x=ox, y=oy, z=1.0)))
        if occluder is None:
            print("WARNING: the occluding vehicle did not spawn; the VRU will not be hidden.")
        else:
            actors.append(occluder)
            print(f"Occluder: {occluder.type_id}")

        sx, sy = sc["vru_start"]
        if sc["vru_type"] == "walker":
            walker_bps = lib.filter("walker.pedestrian.*")
            for i in range(sc["vru_count"]):
                a = spawn_with_retries(world, tagged(walker_bps[i % len(walker_bps)]),
                                       sx + (i % 3) * 1.5, sy + (i // 3) * 1.2, 1.0, -90.0)
                if a is not None:
                    actors.append(a)
                    vrus.append(a)
        else:
            bike_bp = tagged(first_blueprint(lib, ["vehicle.diamondback.century", "vehicle.bh.crossbike",
                                                   "vehicle.gazelle.omafiets"]))
            for i in range(sc["vru_count"]):
                a = spawn_with_retries(world, bike_bp, sx + i * 2.0, sy, 1.0, -90.0)
                if a is not None:
                    actors.append(a)
                    vrus.append(a)
        if not vrus:
            raise RuntimeError("No VRU could be spawned.")
        vru_ids = {a.id for a in vrus}

        def camera_bp(fov):
            bp = tagged(lib.find("sensor.camera.rgb"))
            bp.set_attribute("image_size_x", str(IMG_W))
            bp.set_attribute("image_size_y", str(IMG_H))
            bp.set_attribute("fov", str(fov))
            bp.set_attribute("sensor_tick", str(DETECTION_INTERVAL))
            return bp

        ego_cam = world.spawn_actor(camera_bp(EGO_CAM_FOV), EGO_CAM_OFFSET, attach_to=ego)
        actors.append(ego_cam)
        rsu_cam = world.spawn_actor(camera_bp(RSU_CAM_FOV), RSU_TRANSFORM)
        actors.append(rsu_cam)
        col_sensor = world.spawn_actor(tagged(lib.find("sensor.other.collision")), carla.Transform(), attach_to=ego)
        actors.append(col_sensor)

        ego_cam.listen(make_camera_callback(frames, "ego"))
        rsu_cam.listen(make_camera_callback(frames, "rsu"))
        col_sensor.listen(lambda e: collisions.append(e.other_actor.id if e.other_actor else -1))

        # let everything settle on the ground
        for _ in range(10):
            world.tick()

        cx, cy = sc["conflict"]
        ground_z = estimate_ground_z(world, cx, cy, fallback=ego.get_location().z)
        ext_x, ext_y = ego.bounding_box.extent.x, ego.bounding_box.extent.y

        # Our own driver instead of the Traffic Manager autopilot (the autopilot gave no throttle
        # in synchronous mode). This also makes every run drive exactly the same way.
        driver = LaneFollower(world.get_map(), ego)
        ego_start = ego.get_transform()
        wp0 = world.get_map().get_waypoint(ego_start.location)
        print(f"Ego at ({ego_start.location.x:.1f}, {ego_start.location.y:.1f}), heading {ego_start.rotation.yaw:.0f} deg | "
              f"lane heading {wp0.transform.rotation.yaw:.0f} deg | cruise {CRUISE_SPEED_KMH:.0f} km/h")

        # Bicycles are moved kinematically (physics off, position set every tick): with physics on
        # they can tip over or be pushed off course, which makes runs unrepeatable.
        if sc["vru_type"] != "walker":
            for a in vrus:
                bikes.append((a, a.get_transform()))
                a.set_simulate_physics(False)

        t0 = world.get_snapshot().timestamp.elapsed_seconds
        vru_started = False
        last_det = -1e9
        last_status, stopped_since, stuck_warned = -1e9, None, False
        decision = dict(level="NONE", brake=0.0, speed_cap_kmh=None, source="", threat=None)

        # ---------------- main loop ----------------
        for _ in range(MAX_TICKS):
            world.tick()
            now = world.get_snapshot().timestamp.elapsed_seconds   # shared simulation clock
            t_rel = now - t0

            if not vru_started and t_rel >= sc["vru_start_delay"]:
                for a in vrus:
                    if sc["vru_type"] == "walker":
                        a.apply_control(carla.WalkerControl(direction=carla.Vector3D(0.0, -1.0, 0.0),
                                                            speed=sc["vru_speed"] + random.uniform(-0.3, 0.3)))
                vru_started = True
            for a, tf_start in bikes:   # kinematic bicycle: straight line along its heading
                s = sc["vru_speed"] * max(0.0, t_rel - sc["vru_start_delay"])
                f = tf_start.get_forward_vector()
                a.set_transform(carla.Transform(carla.Location(x=tf_start.location.x + f.x * s,
                                                               y=tf_start.location.y + f.y * s,
                                                               z=tf_start.location.z), tf_start.rotation))

            ego_tf = ego.get_transform()
            v = ego.get_velocity()
            speed = math.hypot(v.x, v.y)
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)
            conflict_dist, _ = to_ego_frame(cx, cy, ego_tf.location.x, ego_tf.location.y, ego_tf.rotation.yaw)

            # ground truth, for metrics only
            truth = [a.get_location() for a in vrus if a.is_alive]
            gaps = [gap_to_ego_body(*to_ego_frame(l.x, l.y, ego_tf.location.x, ego_tf.location.y,
                                                   ego_tf.rotation.yaw), ext_x, ext_y) for l in truth]
            min_gap_now = min(gaps) if gaps else float("inf")

            # ---------------- perception (every DETECTION_INTERVAL) ----------------
            new_ego_frame = None
            if now - last_det >= DETECTION_INTERVAL - 1e-6:
                last_det = now

                # RSU: detect -> world coordinates -> track -> compact message -> publish
                if frames["rsu"] is not None:
                    img, ts, cam_m = frames["rsu"]
                    vis = img.copy()
                    world_dets = []
                    for d in detect_vrus(img):
                        g = pixel_to_ground(d["foot"][0], d["foot"][1], cam_m, RSU_CAM_FOV, ground_z)
                        if g is None:
                            continue
                        world_dets.append(dict(x=g[0], y=g[1], cls=d["cls"], conf=d["conf"]))
                        draw_box(vis, d["box"], f"{d['cls']} ({g[0]:.1f}, {g[1]:.1f})", (0, 255, 0))
                    tracks = rsu_tracker.update(world_dets, ts)
                    for tr in tracks:   # validation: estimate vs. true position
                        if truth:
                            ex, ey = tr["x"] + tr["vx"] * (now - ts), tr["y"] + tr["vy"] * (now - ts)
                            err = min(math.hypot(ex - l.x, ey - l.y) for l in truth)
                            if err < 5.0:
                                loc_errors.append(err)
                    seq += 1
                    payload = encode_vru_message(RSU_ID, seq, ts, tracks)
                    link.publish(payload, seq)
                    if tracks and example_msg is None:
                        example_msg = payload.decode("utf-8")
                    cv2.putText(vis, f"{RSU_ID} | {len(tracks)} VRU(s) | msg #{seq}, {len(payload)} bytes",
                                (10, 25), FONT, 0.6, (0, 255, 255), 2)
                    cv2.imshow("RSU camera", vis)

                # Vehicle: own camera -> world coordinates -> track
                if frames["ego"] is not None:
                    img, ts, cam_m = frames["ego"]
                    vis = img.copy()
                    world_dets = []
                    for d in detect_vrus(img):
                        g = pixel_to_ground(d["foot"][0], d["foot"][1], cam_m, EGO_CAM_FOV, ground_z)
                        if g is None:
                            continue
                        world_dets.append(dict(x=g[0], y=g[1], cls=d["cls"], conf=d["conf"]))
                        draw_box(vis, d["box"], f"{d['cls']} {d['conf']:.2f}", (0, 255, 0))
                    local_tracks = ego_tracker.update(world_dets, ts)
                    new_ego_frame = (vis, cam_m)

            # ---------------- receive, fuse, decide ----------------
            v2x_objs = receiver.poll(now)
            v2x_threats = [assess_threat(o, ego_tf, speed, now, ext_x) for o in v2x_objs]
            local_threats = [assess_threat(o, ego_tf, speed, now, ext_x) for o in local_tracks]

            if events["v2x_first"] is None and any(t["relevant"] for t in v2x_threats):
                events["v2x_first"] = dict(t=t_rel, conflict_dist=conflict_dist, speed_kmh=speed * 3.6)
            if events["local_first"] is None and any(t["relevant"] for t in local_threats):
                events["local_first"] = dict(t=t_rel, conflict_dist=conflict_dist, speed_kmh=speed * 3.6)

            decision = decide(local_threats, v2x_threats, v2x_enabled, speed, ext_x)

            # ---------------- control ----------------
            if decision["level"] != "NONE" and events["reaction"] is None:
                events["reaction"] = dict(t=t_rel, conflict_dist=conflict_dist, speed_kmh=speed * 3.6,
                                          level=decision["level"], source=decision["source"])
                print(f"[{decision['level']}] t={t_rel:.2f} s, source {decision['source']}, "
                      f"speed {speed * 3.6:.1f} km/h, {conflict_dist:.1f} m before the conflict point")
            if decision["level"] == "CAUTION":
                last_caution_t = now

            steer = driver.steer(ego_tf, speed)
            prev_state = state
            if decision["brake"] > 0:
                # Braking manoeuvre requested by the policy
                throttle, brake = 0.0, decision["brake"]
                last_threat_t, hold_brake = now, brake
                state = "BRAKE"
            elif now - last_threat_t < RESUME_HOLD_S:
                # Keep the brake for a moment after the threat disappears
                throttle, brake = 0.0, hold_brake
                state = "HOLD"
            else:
                # Normal driving; after an RSU warning (CAUTION) the target speed is lowered for a while
                slow = now - last_caution_t < RESUME_HOLD_S
                target = V2X_TARGET_SPEED_KMH if slow else CRUISE_SPEED_KMH
                throttle, brake = driver.speed_control(speed, target)
                state = "SLOW" if slow else "CRUISE"
                if prev_state in ("BRAKE", "HOLD"):
                    print(f"[RESUME] t={t_rel:.2f} s")
            ego.apply_control(carla.VehicleControl(throttle=throttle, steer=steer, brake=brake))
            applied_brake = brake

            # ---------------- display ----------------
            world.debug.draw_string(ego_tf.location + carla.Location(z=3.0),
                                    f"{mode} | {decision['level']}", draw_shadow=True,
                                    color=LEVEL_COLOR_CARLA[decision["level"]], life_time=FIXED_DT * 1.5)
            if v2x_enabled:
                for th in v2x_threats:
                    world.debug.draw_point(carla.Location(x=th["x_now"], y=th["y_now"], z=ground_z + 1.0),
                                           size=0.15, color=carla.Color(255, 140, 0), life_time=FIXED_DT * 1.5)
            if new_ego_frame is not None:
                vis, cam_m = new_ego_frame
                draw_ego_hud(vis, decision, v2x_enabled, speed * 3.6, v2x_threats, cam_m, ground_z, ext_x, link.mode)
                cv2.imshow("Ego camera", vis)
            cv2.waitKey(1)

            # ---------------- diagnostics ----------------
            if STATUS_EVERY_S and t_rel - last_status >= STATUS_EVERY_S:
                last_status = t_rel
                near = min(truth, key=lambda l: l.distance(ego_tf.location)) if truth else None
                vru_txt = f"VRU at ({near.x:.1f}, {near.y:.1f})" if near else "no VRU"
                print(f"  t={t_rel:5.1f}s | {speed * 3.6:5.1f} km/h | {state:6s} | "
                      f"throttle {throttle:.2f} steer {steer:+.2f} brake {brake:.2f} | "
                      f"alert {decision['level']} {decision['source']} | "
                      f"{conflict_dist:5.1f} m to conflict point | {vru_txt}")
            if speed < 0.3:
                stopped_since = now if stopped_since is None else stopped_since
                if not stuck_warned and now - stopped_since > 3.0:
                    stuck_warned = True
                    if decision["level"] != "NONE":
                        th = decision["threat"]
                        why = (f"our policy is braking ({decision['level']}, source {decision['source']}): "
                               f"{th['cls']} at {th['lon']:.1f} m ahead, {th['lat']:.1f} m sideways")
                    elif state == "HOLD":
                        why = "the brake is still held after a threat"
                    else:
                        why = (f"the driver gives throttle {throttle:.2f} but the car does not move "
                               f"(blocked by an object, or spawned inside something?)")
                    print(f"[STUCK] Car has been stopped for 3 s: {why}")
            else:
                stopped_since = None

            log.append(dict(t=t_rel, speed_kmh=speed * 3.6, conflict_dist=conflict_dist,
                            min_gap=min_gap_now, level=decision["level"], brake=applied_brake))

            if conflict_dist < -20.0:     # the ego has passed the crossing
                break

    finally:
        for s in (ego_cam, rsu_cam, col_sensor):
            try:
                if s is not None:
                    s.stop()
            except Exception:
                pass
        try:
            world.tick()                  # let in-flight sensor callbacks finish
        except Exception:
            pass
        safe_destroy(list(reversed(actors)))
        try:
            world.tick()
        except Exception:
            pass
        world.apply_settings(original_settings)
        cv2.destroyAllWindows()
        for _ in range(5):
            cv2.waitKey(1)

    # ---------------- metrics ----------------
    df = pd.DataFrame(log)
    nan = float("nan")
    min_gap = float(df["min_gap"].min()) if len(df) else nan
    collision = any(i in vru_ids for i in collisions) or (min_gap < COLLISION_GAP)
    near_miss = (not collision) and (min_gap < NEAR_MISS_GAP)

    approach = df[df["conflict_dist"] > 0] if len(df) else df
    react = events["reaction"]
    if react is not None:
        after = approach[approach["t"] >= react["t"]]
        v_react = react["speed_kmh"]
        v_min = float(after["speed_kmh"].min()) if len(after) else v_react
        speed_reduction = v_react - v_min
    else:
        v_react = v_min = nan
        speed_reduction = 0.0
    speed_at_closest = float(df.loc[df["min_gap"].idxmin(), "speed_kmh"]) if len(df) else nan

    v2x_first, local_first = events["v2x_first"], events["local_first"]
    lead_time = local_first["t"] - v2x_first["t"] if (v2x_first and local_first) else nan
    lead_dist = v2x_first["conflict_dist"] - local_first["conflict_dist"] if (v2x_first and local_first) else nan
    st = receiver.stats

    summary = dict(
        scenario=sc["id"], name=sc["name"], mode=mode,
        collision=collision, near_miss=near_miss, min_gap_m=round(min_gap, 2),
        v2x_warning_t_s=v2x_first["t"] if v2x_first else nan,
        local_detection_t_s=local_first["t"] if local_first else nan,
        warning_lead_time_s=lead_time, warning_lead_dist_m=lead_dist,
        reaction_source=react["source"] if react else "none",
        reaction_dist_to_conflict_m=react["conflict_dist"] if react else nan,
        speed_at_reaction_kmh=v_react, min_speed_before_conflict_kmh=v_min,
        speed_reduction_kmh=speed_reduction, speed_at_closest_kmh=speed_at_closest,
        rsu_loc_error_m=float(np.mean(loc_errors)) if loc_errors else nan,
        v2x_msgs=st["delivered"], v2x_lost=st["lost"],
        v2x_msg_bytes_mean=float(np.mean(st["bytes"])) if st["bytes"] else nan,
        v2x_latency_sim_ms=float(np.mean(st["sim_latency"]) * 1000) if st["sim_latency"] else nan,
        v2x_transport_wall_ms=float(np.mean(st["wall_ms"])) if st["wall_ms"] else nan,
        link=link.mode, example_msg=example_msg)
    lead_txt = f"{lead_time:.2f} s" if not math.isnan(lead_time) else "n/a"
    print(f"-> collision={collision}, near miss={near_miss}, min gap={min_gap:.2f} m, warning lead time={lead_txt}")
    return summary, df


print("Scenario runner defined.")

## Cell 8: Occlusion scenarios and results

| ID | Occlusion scenario | VRU | Weather |
|---|---|---|---|
| **S1** | pedestrian steps out from behind a parked van | 1 pedestrian | clear |
| **S2** | cyclist rides out from behind a parked truck | 1 cyclist | clear |
| **S3** | group of pedestrians behind the van in heavy rain | 15 pedestrians | heavy rain |

Each scenario runs twice: **baseline** (vehicle uses only its own camera) and **V2X** (vehicle also uses the RSU warning). Everything else is identical: same spawn points, detector, thresholds and driver. So the difference between the two runs is caused only by the V2X warning.

To run fewer scenarios, edit `RUN_SCENARIOS`. The VRUs start moving 1.5 s (pedestrians) or 2.0 s (cyclist) after the car, so that they actually meet the car at the crossing. If a baseline run shows no near miss or collision, adjust `vru_start_delay`.

The results are saved to `results.csv`, and each run's time series to `log_<scenario>_<mode>.csv`.

In [ ]:
# ==============================================================================
# CELL 8: OCCLUSION SCENARIOS - RUN WITH AND WITHOUT ASSISTANCE
# ==============================================================================
SCENARIOS = [
    dict(id="S1", name="Pedestrian behind parked van", weather="ClearNoon",
         occluder=["vehicle.mercedes.sprinter"], occluder_pos=(127.0, 135.6),
         vru_type="walker", vru_count=1, vru_start=(130.0, 145.0), vru_speed=3.5, vru_start_delay=1.5,
         conflict=(130.5, 133.0)),
    dict(id="S2", name="Cyclist behind parked truck", weather="ClearNoon",
         occluder=["vehicle.carlamotors.carlacola", "vehicle.ford.ambulance", "vehicle.volkswagen.t2_2021",
                   "vehicle.volkswagen.t2", "vehicle.mercedes.sprinter"],
         occluder_pos=(126.5, 135.6),
         vru_type="bicycle", vru_count=1, vru_start=(130.5, 146.0), vru_speed=4.5, vru_start_delay=2.0,
         conflict=(130.5, 133.0)),
    dict(id="S3", name="Pedestrian group behind van, heavy rain", weather="HardRainSunset",
         occluder=["vehicle.mercedes.sprinter"], occluder_pos=(127.0, 135.6),
         vru_type="walker", vru_count=15, vru_start=(130.0, 145.0), vru_speed=3.5, vru_start_delay=1.5,
         conflict=(131.5, 133.0)),
]
RUN_SCENARIOS = ["S1", "S2", "S3"]
# vru_start_delay sets when the VRU starts moving (seconds after the car starts). It decides whether the
# car and the VRU actually meet at the crossing: too early and the VRU has crossed before the car arrives
# (no conflict, both runs identical); too late and the car has already passed. The values above were
# chosen so that the baseline run ends in a near miss or collision. If a run shows no conflict, adjust them.

# Load the map once and reuse it for every run (reloading between runs crashed the kernel)
world = client.get_world()
if not world.get_map().name.endswith(MAP_NAME):
    world = client.load_world(MAP_NAME)
    time.sleep(3.0)
cleanup_leftovers(world)

link = V2XLink()
results, logs = [], {}
try:
    for sc in SCENARIOS:
        if sc["id"] not in RUN_SCENARIOS:
            continue
        for v2x_enabled in (False, True):
            summary, log_df = run_scenario(world, sc, v2x_enabled, link)
            results.append(summary)
            logs[(sc["id"], summary["mode"])] = log_df
            log_df.to_csv(f"log_{sc['id']}_{summary['mode']}.csv", index=False)
finally:
    link.close()

res = pd.DataFrame(results)
res.drop(columns=["example_msg"]).to_csv("results.csv", index=False)

# ---------------- comparison table: baseline vs V2X per scenario ----------------
rows = []
for sid in res["scenario"].unique():
    b = res[(res.scenario == sid) & (res["mode"] == "baseline")]
    a = res[(res.scenario == sid) & (res["mode"] == "V2X")]
    if b.empty or a.empty:
        continue
    b, a = b.iloc[0], a.iloc[0]

    def outcome(r):
        return "COLLISION" if r.collision else ("NEAR MISS" if r.near_miss else "SAFE")

    rows.append({
        "Scenario": f"{sid}: {b['name']}",
        "Warning lead time [s]": round(b.warning_lead_time_s, 2),
        "Lead distance [m]": round(b.warning_lead_dist_m, 1),
        "Reaction distance to conflict, baseline / V2X [m]":
            f"{b.reaction_dist_to_conflict_m:.1f} / {a.reaction_dist_to_conflict_m:.1f}",
        "Speed reduction, baseline / V2X [km/h]": f"{b.speed_reduction_kmh:.1f} / {a.speed_reduction_kmh:.1f}",
        "Speed at closest approach, baseline / V2X [km/h]":
            f"{b.speed_at_closest_kmh:.1f} / {a.speed_at_closest_kmh:.1f}",
        "Min gap, baseline / V2X [m]": f"{b.min_gap_m:.2f} / {a.min_gap_m:.2f}",
        "Outcome, baseline / V2X": f"{outcome(b)} / {outcome(a)}",
        "Near miss avoided": "YES" if (b.collision or b.near_miss) and not (a.collision or a.near_miss) else "no",
    })
comparison = pd.DataFrame(rows)
comparison.to_csv("comparison.csv", index=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
print("\n=========================== BASELINE vs V2X ===========================")
print(comparison.T.to_string(header=False))
print("\n=========================== V2X LINK ===========================")
v2x_rows = res[res["mode"] == "V2X"]
print(v2x_rows[["scenario", "link", "v2x_msgs", "v2x_lost", "v2x_msg_bytes_mean",
                "v2x_latency_sim_ms", "v2x_transport_wall_ms", "rsu_loc_error_m"]].to_string(index=False))
examples = res["example_msg"].dropna()
if len(examples):
    print("\nExample V2X message:\n", examples.iloc[0])

## Cell 9: Speed profiles

Speed against distance to the conflict point, baseline (dashed) and V2X (solid). The vertical lines mark where each run first reacted. With V2X, the reaction should come earlier (farther from the conflict point) and the speed near the crossing should be lower. The figures are saved as `speed_profile_<scenario>.png` for the report.

In [ ]:
# ==============================================================================
# CELL 9: SPEED PROFILES FOR THE REPORT
# ==============================================================================
import matplotlib.pyplot as plt

for sid in res["scenario"].unique():
    fig, ax = plt.subplots(figsize=(8, 4))
    for mode, style in (("baseline", "--"), ("V2X", "-")):
        df = logs.get((sid, mode))
        if df is None or df.empty:
            continue
        ax.plot(df["conflict_dist"], df["speed_kmh"], style, label=mode)
        r = res[(res.scenario == sid) & (res["mode"] == mode)].iloc[0]
        if not math.isnan(r.reaction_dist_to_conflict_m):
            ax.axvline(r.reaction_dist_to_conflict_m, linestyle=style, alpha=0.4)
            ax.annotate(f"{mode}: reaction ({r.reaction_source})",
                        (r.reaction_dist_to_conflict_m, r.speed_at_reaction_kmh),
                        textcoords="offset points", xytext=(5, 8), fontsize=8)
    ax.set_xlabel("Distance to conflict point [m]")
    ax.set_ylabel("Speed [km/h]")
    ax.set_title(f"{sid}: {SCENARIOS[[s['id'] for s in SCENARIOS].index(sid)]['name']}")
    ax.invert_xaxis()          # the vehicle drives from left (far) to right (conflict point)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"speed_profile_{sid}.png", dpi=150)
    plt.show()